<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/ml/notebooks/c5_l9.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C5-L9 · Monitoreo: IC y drift
Pulso (IC rolling) y fiebre (z-shift de features): el semáforo que decide si la señal opera, reduce o se apaga.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/ml/data/c5_l9.csv'
try:
    df = pd.read_csv(URL)
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data/c5_l9.csv'), Path('data/c5_l9.csv'), Path('c5_l9.csv')]:
        if cand.exists():
            df = pd.read_csv(cand); break
    print('Fuente: local')
print(df.shape)
print(df.head(5).to_string(index=False))

In [ ]:
df['ret'] = df['close'].pct_change()
df['rango'] = (df['high']-df['low'])/df['close']
for k in (1, 2, 3, 5):
    df[f'lag_{k}'] = df['ret'].shift(k)
df['mom5'] = df['close']/df['close'].shift(5) - 1
data = df.dropna().reset_index(drop=True)
data['fwd'] = data['ret'].shift(-1)
data = data.iloc[:-1].reset_index(drop=True)
def spearman(a, b):
    ra = pd.Series(np.asarray(a, float)).rank().values
    rb = pd.Series(np.asarray(b, float)).rank().values
    return float(np.corrcoef(ra, rb)[0, 1])
ic = spearman(data['mom5'].values, data['fwd'].values)
print(f'filas: {len(data)}  IC global: {ic:.4f}')
assert np.isfinite(ic) and abs(ic) < 0.6

In [ ]:
V = 20
ic_roll = np.array([spearman(data['mom5'].values[i:i+V], data['fwd'].values[i:i+V]) for i in range(len(data)-V)])
print(f'IC rolling medio: {np.nanmean(ic_roll):.4f} | ventanas+ : {np.nanmean(ic_roll > 0):.0%} | n={len(ic_roll)}')

In [ ]:
train = data.iloc[:len(data)//2]; rec = data.iloc[-20:]
def zshift(col):
    return float((rec[col].mean() - train[col].mean())/train[col].std())
z_rango, z_lag = zshift('rango'), zshift('lag_1')
print(f'z-shift rango={z_rango:.2f} lag_1={z_lag:.2f}')
semaforo = ('ROJO apaga' if abs(z_rango) > 1.5 and abs(z_lag) > 1.5 else
    ('AMARILLO reduce' if abs(z_rango) > 1.5 or abs(z_lag) > 1.5 or np.nanmean(ic_roll) <= 0 else 'VERDE opera'))
print('Semáforo:', semaforo)

In [ ]:
assert len(ic_roll) > 0 and np.isfinite(ic_roll).all()
assert np.isfinite([z_rango, z_lag]).all()
print(f'OK L9: IC={ic:.4f} rolling={np.nanmean(ic_roll):.4f} semáforo={semaforo}')